# E06_Hunter3D — PPO с формированием награды

**Урок:** 2.2 «PPO с нуля», 2.4 «Reward Shaping» и проект-2 «3D-охотник»
курса [cyber-unity-learn](https://rl-cuber-unity-code.com/)
**Что реализуется:** PPO на среде навигации с потенциальным формированием награды.
**Ожидаемый результат:** **≥ 80 %** пойманных целей на 20 эпизодах оценки.
**Время обучения:** около 20 минут на сид (`QUICK_RUN = True` — около двух минут).

## Что здесь принципиально нового

Метод тот же, что в `E04_BallBalance__ppo.ipynb`, — новое в **среде**, и оба
новшества меняют то, как за обучением следят.

**1. Два сенсора разной природы.** `RayPerceptionSensor3D` даёт «зрение»
(15 чисел: обход препятствий, факт видимости цели), `VectorSensor` —
проприоцепцию (10 чисел: гладкое наведение). Python видит их как `obs_0`
и `obs_1`; политика получает склейку 15 + 10 = 25. Порядок склейки задаёт
не наш код, а ML-Agents: сенсоры сортируются по имени, и `RayPerceptionSensor`
идёт перед `VectorSensor_size10`. Тот же порядок обязан быть в графе ONNX,
иначе Unity подаст сети перепутанные входы — и не сообщит об этом.

**2. Потенциальное формирование награды (PBRS).** «Поймал / не поймал»
на горизонте 200 решений — слишком редкий сигнал. Плотный даётся в форме
$F(s,s') = \gamma\Phi(s') - \Phi(s)$, единственной, которая доказуемо
не меняет оптимальную политику (теорема Ына, 1999).

## Отсюда — главное следствие для приёмки

Недисконтированная сумма награды в этой среде **не является** мерой качества.
Она содержит остаточное смещение $(\gamma-1)\sum_t \Phi(s_{t+1})$, которое
зависит от того, где агент провёл эпизод, а не от того, чем эпизод кончился:
провал у дальней стенки стоит +0.41, провал рядом с целью −0.65
(T-15 в `docs/07_TROUBLESHOOTING.md`).

Поэтому и оценка, и критерий приёмки здесь идут **по исходу эпизода**:
`success_rule = "terminated"`, то есть по доле пойманных целей.

## Требования

```powershell
python scripts\build_env.py E06_Hunter3D
```

## 1. Окружение

In [ ]:
# --- 1. Окружение -------------------------------------------------------
import platform
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "python" / "labrl").is_dir():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("не найден корень репозитория")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "python"))

import numpy as np
import torch
import torch.nn as nn
import mlagents_envs

from labrl.utils.seeding import resolve_device, set_global_seed

SEED = 0
set_global_seed(SEED)
DEVICE = resolve_device("auto")

print(f"python        {platform.python_version()}")
print(f"torch         {torch.__version__}  CUDA {torch.version.cuda}")
print(f"mlagents_envs {mlagents_envs.__version__}")
print(f"устройство    {DEVICE}")
print(f"сид           {SEED}")

## 2. Конфигурация

Значения совпадают с `configs/E06_Hunter3D__ppo.yaml`: ноутбук и скрипт обязаны
обучать одинаково, иначе учебный и рабочий пути разойдутся.

Отличия от `E04`, и каждое имеет причину:

| Параметр | E04 | E06 | Почему |
|---|---|---|---|
| `rollout_steps` | 128 | 512 | навигация даёт более редкий сигнал; батч 4096 переходов устойчивее |
| `entropy_coef` | 0.001 | 0.01 | нужна разведка: цель надо сначала найти |
| `total_steps` | 100 000 | 160 000 | при 80 000 кривая ещё росла (90 / 75 / 70 % на трёх сидах) |
| `success_rule` | по награде | `terminated` | см. вводный раздел |

**`gamma` обязана совпадать с `shapingGamma` агента в Unity (0.99).** Теорема
Ына доказана для одного и того же $\gamma$; при расхождении формирование
награды перестаёт быть безопасным преобразованием задачи.

In [ ]:
QUICK_RUN = True   # True — быстрая проверка (~2 мин); False — полное обучение (~20 мин)

from labrl.algos.ppo import PPOConfig
from labrl.eval.success import TERMINATED
from labrl.train.onpolicy import OnPolicyTrainConfig
from labrl.utils.schedules import LinearSchedule

TOTAL_STEPS = 16_000 if QUICK_RUN else 160_000

ALGO_CFG = PPOConfig(
    gamma=0.99,              # ОБЯЗАНА совпадать с shapingGamma в HunterAgent.cs
    gae_lambda=0.95,
    learning_rate=3e-4,
    clip_range=0.2,
    epochs=4,
    minibatch_size=256,
    target_kl=0.0,
    value_coef=0.5,
    entropy_coef=0.01,       # выше, чем в E04: цель надо сначала найти
    max_grad_norm=0.5,
    normalize_advantage=True,
)

TRAIN_CFG = OnPolicyTrainConfig(
    total_steps=TOTAL_STEPS,
    rollout_steps=512,           # 512 шагов × 8 арен = 4096 переходов на обновление
    eval_every_steps=max(1, TOTAL_STEPS // 8),   # 20 000 на полном бюджете
    eval_episodes=20,
    success_rule=TERMINATED,     # успех = цель поймана, а не «награда выше порога»
    success_threshold=0.0,       # правилом terminated не используется
)

# Расписание НЕ сжимается вместе с сокращённым прогоном (T-10).
LEARNING_RATE = LinearSchedule(start=3e-4, end=0.0, decay_steps=160_000)

updates = TOTAL_STEPS // TRAIN_CFG.rollout_steps
print(f"бюджет: {TOTAL_STEPS} шагов сбора, обновлений: {updates}, "
      f"градиентных шагов: {updates * ALGO_CFG.epochs * (512 * 8 // ALGO_CFG.minibatch_size)}")

## 3. Подключение среды

In [ ]:
from labrl.envs.registry import get as get_env
from labrl.envs.unity_env import open_unity_env
from labrl.envs.vec_unity_env import VecUnityEnv

ENV_ID = "E06_Hunter3D"
BUILD = get_env(ENV_ID).build_path
if not BUILD.is_file():
    raise FileNotFoundError(f"нет билда {BUILD}\nСоберите: python scripts/build_env.py {ENV_ID}")

handle = open_unity_env(ENV_ID, build_path=BUILD, seed=SEED, time_scale=20.0, no_graphics=True)
vec = VecUnityEnv(handle)
obs = vec.reset()
print(handle.describe())
print(f"\nарен (параллельных сред): {vec.num_envs}")

## 4. Инспекция пространств

Здесь наблюдений **два**, и проверять надо не только суммарную размерность,
но и порядок: перепутанные местами сенсоры дают ту же сумму 25 и полностью
другую задачу.

In [ ]:
OBS_DIM = vec.single_obs_dim          # склейка всех сенсоров: 15 + 10
ACTION_DIM = int(vec.action_spec.continuous_size)

print(f"obs_shapes:        {vec.obs_shapes}")
print(f"obs_dim (склейка): {OBS_DIM}")
print(f"непрерывных:       {ACTION_DIM}")
print(f"дискретные ветки:  {tuple(vec.action_spec.discrete_branches)}")
print(f"\nobs_0 (лучи, 15):  {np.array2string(obs[0][0], precision=2)}")
print(f"obs_1 (вектор, 10): {np.array2string(obs[1][0], precision=2)}")
print("  obs_1: [0:2] вектор на цель, [2] дистанция, [3] угол,")
print("         [4:6] своя скорость, [6] угловая скорость, [7:9] «вперёд», [9] доля времени")

# Значения из строки-контракта ENV_SPEC.md:
# <!-- validator: obs_size=10; discrete_branches=; continuous_size=2; max_step=1000 -->
assert [tuple(s) for s in vec.obs_shapes] == [(15,), (10,)], (
    f"порядок или размер сенсоров не тот: {vec.obs_shapes}; "
    "ML-Agents сортирует сенсоры по имени — RayPerceptionSensor перед VectorSensor_size10")
assert OBS_DIM == 25, f"obs_dim {OBS_DIM} != 25"
assert ACTION_DIM == 2, f"непрерывных действий {ACTION_DIM} != 2 (ENV_SPEC.md)"
assert len(vec.action_spec.discrete_branches) == 0
print("\nразмерности и порядок сенсоров совпадают с ENV_SPEC.md")

## 5. Нейросети

Актор и критик — те же, что в `E04`, только вход шире (25 вместо 8).
Склейку сенсоров делает обёртка среды (`VecUnityEnv.flatten_obs`), поэтому
сеть остаётся обычным MLP: разделять признаки по происхождению здесь незачем,
оба сенсора — плотные числовые векторы одного масштаба.

Активация `tanh`: задача управления, гладкая политика ведёт себя лучше.

In [ ]:
from labrl.nets.gaussian_policy import GaussianPolicyNetwork


class MyPolicy(GaussianPolicyNetwork):
    """Актор: obs (B, 25) -> среднее действия (B, 2) в сырых единицах."""

    def __init__(self, obs_dim: int, action_dim: int, hidden: int = 128):
        super().__init__(obs_dim, action_dim, (hidden, hidden), "tanh", log_std_init=-0.3)


class MyCritic(nn.Module):
    """Критик: obs (B, 25) -> V(s) (B, 1). В Unity не уезжает."""

    def __init__(self, obs_dim: int, hidden: int = 128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs):
        return self.body(obs)


policy_net = MyPolicy(OBS_DIM, ACTION_DIM)
value_net = MyCritic(OBS_DIM)

with torch.no_grad():
    probe = torch.zeros(3, OBS_DIM)
    assert policy_net(probe).shape == (3, ACTION_DIM), "актор обязан выдавать (B, action_dim)"
    assert value_net(probe).shape == (3, 1), "критик обязан выдавать (B, 1)"

print(policy_net)
print(f"\nпараметров актора:  {sum(p.numel() for p in policy_net.parameters()):,}")
print(f"параметров критика: {sum(p.numel() for p in value_net.parameters()):,}")

## 6. Теория: почему формирование награды безопасно именно в этой форме

Наивная плотная награда «+0.1 за каждое сокращение дистанции» создаёт
классический reward hacking: отойти назад (бесплатно) и подойти снова (+0.1)
выгоднее, чем дойти до цели один раз.

Потенциальное формирование (Ng, Harada, Russell, 1999) задаёт добавку через
функцию **состояния**:

$$\Phi(s) = -\frac{d(s)}{d_{\max}}, \qquad
F(s, s') = \gamma\,\Phi(s') - \Phi(s)$$

Дисконтированная сумма такой добавки телескопируется:

$$\sum_t \gamma^t F(s_t, s_{t+1}) = \gamma^T \Phi(s_T) - \Phi(s_0)$$

— она зависит только от концов траектории, а не от того, как агент шёл.
Поэтому множество оптимальных политик не меняется: любой «круг почёта»
приносит ровно ноль.

**Два условия, без которых теорема не работает.**

1. $\Phi(\text{терминального}) = 0$. В коде агента это ветка
   `caught ? 0f : Potential()`. В справочном коде урока условие опущено.
2. $\gamma$ формирования = $\gamma$ алгоритма. Оба равны 0.99.

**И одно свойство, которое остаётся даже при выполненных условиях:**
недисконтированная сумма (та, что в TensorBoard) телескопируется не полностью,
и в ней остаётся смещение — см. вводный раздел и T-15.

Ниже — исходный код обновления PPO, тот самый, что выполняется при обучении.

In [ ]:
import inspect

from labrl.algos.ppo import PPO

print(inspect.getsource(PPO.update))

## 7. Цикл обучения

Цикл — общий `labrl.train.onpolicy.train_on_policy`, тот же, что у A2C и PPO
в `E04`. Он же вызывает `vec.flatten_obs`, склеивая два сенсора в один вход.

In [ ]:
from labrl.logging.run_dir import create_run_dir
from labrl.logging.tb_logger import TBLogger
from labrl.train.onpolicy import train_on_policy

algo = PPO(policy_net, value_net, ALGO_CFG, device=DEVICE, seed=SEED)
run = create_run_dir(ENV_ID, "ppo", SEED)
run.write_env_info({**handle.env_info(), "num_envs": vec.num_envs, "seed": SEED,
                    "total_steps": TOTAL_STEPS, "quick": QUICK_RUN})
logger = TBLogger(run.tb)

print(f"каталог прогона: {run.root}\n")

result = train_on_policy(
    vec=vec, algo=algo, logger=logger, cfg=TRAIN_CFG, seed=SEED,
    lr_schedule=LEARNING_RATE,
    on_eval=lambda step, r, s: print(f"  шаг {step:>7}: награда {r:>7.3f}, поймано {s:.0%}"),
)

missing = logger.missing_required_tags()
logger.dump_metrics(run.metrics_json, extra={"seed": SEED, "missing_required_tags": list(missing)})
logger.close()

print(f"\nэпизодов: {len(result.episode_returns)}, переходов: {result.transitions}, "
      f"обновлений: {result.updates}")
print(f"время: {result.wall_time:.1f} с")
print(f"обязательные теги схемы 11.2 без данных: {missing or 'нет'}")

## 8. Оценка и кривая обучения

На втором графике две линии — доля пойманных целей и средняя награда.
Смотреть надо на первую: именно она является мерой качества в этой среде.
Награда показана рядом, чтобы было видно, как расходятся масштабы.

In [ ]:
import matplotlib
matplotlib.use("Agg")   # ноутбук должен исполняться и без графического бэкенда
import matplotlib.pyplot as plt

returns = np.array(result.episode_returns)
window = max(1, len(returns) // 50)
smoothed = np.convolve(returns, np.ones(window) / window, mode="valid")

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].plot(smoothed)
axes[0].set_title(f"Награда за эпизод (скользящее среднее по {window})")
axes[0].set_xlabel("эпизод"); axes[0].set_ylabel("награда"); axes[0].grid(alpha=0.3)

steps = [s for s, _, _ in result.eval_history]
success = [s for _, _, s in result.eval_history]
rewards = [r for _, r, _ in result.eval_history]
axes[1].plot(steps, success, marker="o", color="tab:green", label="доля пойманных")
axes[1].axhline(0.8, color="k", ls="--", lw=1, label="критерий приёмки")
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_title("Eval/Success Rate — мера качества"); axes[1].set_xlabel("шаг сбора")
axes[1].grid(alpha=0.3); axes[1].legend()

axes[2].plot(steps, rewards, marker="o", color="tab:orange")
axes[2].set_title("Eval/Mean Reward — со смещением PBRS")
axes[2].set_xlabel("шаг сбора"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"итоговая оценка: поймано {result.last_eval_success:.0%}, "
      f"награда {result.last_eval_reward:.3f}")

# Критерий приёмки проверяется только на ПОЛНОМ бюджете: сокращённый прогон —
# смоук-тест конвейера, на нём метод не обязан обучиться.
THRESHOLD = 0.80   # блок success_criteria конфига, метрика Eval/Success Rate

if QUICK_RUN:
    print(f"быстрый прогон: критерий приёмки ({THRESHOLD:.0%}) не проверяется — "
          "проверялась работоспособность конвейера")
else:
    assert result.last_eval_success >= THRESHOLD, (
        f"не достигнут критерий приёмки: {result.last_eval_success:.0%} < {THRESHOLD:.0%}")
    print(f"критерий приёмки достигнут: {result.last_eval_success:.0%} >= {THRESHOLD:.0%}")

## 9. Экспорт в ONNX и верификация

Экспортируется актор; критик в Unity не нужен.

Новое по сравнению с `E04`: наблюдений два, поэтому в графе будет **два входа**
(`obs_0` и `obs_1`), и обёртка экспорта склеивает их в том же порядке, в каком
их отдаёт среда. Проверка ниже сравнивает выход настоящего графа с
`algo.deterministic_action()` — если бы порядок разошёлся, равенство сломалось
бы именно здесь, а не через час в Unity.

In [ ]:
import onnxruntime as ort

from labrl.export.onnx_export import ActionSpecLite, export_policy_to_onnx
from labrl.export.onnx_verify import verify_onnx_model

# Наблюдения из реального распределения среды (требование 10.5), а не шум.
collected = [[] for _ in vec.obs_shapes]
obs_now = vec.reset()
while sum(len(c) for c in collected[0]) < 64:
    for sensor, batch in enumerate(obs_now):
        collected[sensor].append(batch.copy())
    obs_now = vec.step(algo.deterministic_action(vec.flatten_obs(obs_now))).obs
sample = [np.concatenate(parts, axis=0)[:64] for parts in collected]

spec = ActionSpecLite(continuous_size=ACTION_DIM)
policy = algo.policy_module()
onnx_path = export_policy_to_onnx(policy, spec, vec.obs_shapes, run.onnx / "policy.onnx")

verification = verify_onnx_model(onnx_path, policy, spec, vec.obs_shapes, sample_obs=sample)
print(verification.report())
verification.raise_if_failed()

# Python и граф обязаны давать одно и то же действие.
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
graph_action = session.run(["deterministic_continuous_actions"],
                           {"obs_0": sample[0], "obs_1": sample[1]})[0]
python_action = algo.deterministic_action(np.concatenate(sample, axis=1))
gap = float(np.abs(graph_action - python_action).max())
print(f"\nmax|ONNX - Python| по действию: {gap:.3e}")
assert gap < 1e-5, "оценка в Python и граф расходятся — инференс в Unity будет другим"

if not QUICK_RUN:
    # Модель проекта обновляется только полным прогоном: быстрый затёр бы её
    # смоук-версией (T-14 в docs/07_TROUBLESHOOTING.md).
    model_dest = (REPO_ROOT / "unity" / "MLAgentsLab" / "Assets" / "Envs" / ENV_ID /
                  "Models" / f"{ENV_ID}_ppo.onnx")
    model_dest.parent.mkdir(parents=True, exist_ok=True)
    model_dest.write_bytes(onnx_path.read_bytes())
    print(f"модель скопирована в проект Unity: {model_dest.relative_to(REPO_ROOT)}")
else:
    print("быстрый прогон: модель в проект Unity НЕ копируется (T-14)")

vec.close()

## 10. Возврат в Unity

### Автоматически (то, чем проверяется приёмка по 10.6)

```powershell
python scripts\check_inference.py --config configs\E06_Hunter3D__ppo.yaml --python-reward <оценка из раздела 8>
```

Скрипт соберёт билд с назначенной моделью и `Behavior Type = Inference Only`,
прогонит 20 эпизодов и сравнит награду с Python-оценкой. Порог — 0.8 от неё.

### Вручную (чтобы увидеть глазами)

1. Открыть `unity/MLAgentsLab` в Unity 6000.5.4f1.
2. Открыть `Assets/Envs/E06_Hunter3D/Scenes/E06_Hunter3D.unity`.
3. Выделить модель в Project → **Tools → RL → Check ONNX Contract**.
4. У агента `Hunter`: `Model` = импортированная модель,
   `Behavior Type` = `Inference Only`, галочка `Deterministic Inference`.
5. Play: охотник объезжает препятствия и идёт к цели; при касании эпизод
   начинается заново с новых позиций.

### Если награда в Unity заметно ниже

Для этой среды порядок разбора начинается не с диапазона действий,
а с **порядка сенсоров**: `obs_0` — лучи, `obs_1` — вектор. Дальше — общий
список причин в `docs/07_TROUBLESHOOTING.md`.

### Ручная проверка среды с клавиатуры

`Behavior Type` = `Heuristic Only`, клавиши **↑/↓** — тяга, **←/→** — поворот.